# U-Net Stability Experiments 1
**WorldView-3 | 16-bit 4-band (R,G,B,NIR1) | 512×512 chips | H100 NVL**

Stability experiments testing reduced augmentation intensity and BS=64 + gradient clipping.
All experiments use fold 1 of a 5-fold split, WD=1e-3, warm-up cosine scheduler.

**NOTE: Do not re-run — results saved to results/unet_training_v3/sweep_bs64_test/**

- Cell 1: WaterChipDatasetV3 — Reduced Augmentation (superseded experiment)
- Cell 2: WaterChipDatasetV3 — Full Augmentation (final version)
- Cell 3: Stability Test 0 — BS=64 + Gradient Clipping + Tversky

In [3]:
# ── Cell 1: WaterChipDatasetV3 — Reduced Augmentation (SUPERSEDED) ──
# NOTE: Do not re-run — retained as historical record only.
# Same on-the-fly approach, but with lighter, more conservative augmentations.
# Goal: reduce per-epoch distribution shift that may be destabilising training.
#
# Changes vs previous version:
# - Per-band jitter reduced from ±20% to ±10%
# - Band dropout REMOVED entirely (was zeroing a whole spectral band — too extreme)
# - Shadow simulation softened (less darkening, smaller regions)
# - Sun glint simulation softened (less brightening)
# - Lower probabilities overall, so fewer augmentations stack per chip

import torch, random, cv2
import numpy as np
import rasterio
from pathlib import Path
from torch.utils.data import Dataset

class WaterChipDatasetV3(Dataset):
    def __init__(self, chip_paths, masks_dir, mean, std, augment=False):
        self.chip_paths = chip_paths
        self.masks_dir  = Path(masks_dir)
        self.mean = torch.tensor(mean).view(-1,1,1)
        self.std  = torch.tensor(std).view(-1,1,1)
        self.augment = augment

    def __len__(self): return len(self.chip_paths)

    def __getitem__(self, idx):
        cp = self.chip_paths[idx]
        mp = self.masks_dir / cp.name
        with rasterio.open(cp) as src:
            image = np.stack([
                src.read(5), src.read(3), src.read(2), src.read(7)
            ]).astype(np.float32)
        with rasterio.open(mp) as src:
            mask = (src.read(1) > 0).astype(np.float32)

        if self.augment:
            image, mask = self._augment(image, mask)

        image = (torch.from_numpy(image) - self.mean) / self.std
        return image, torch.from_numpy(mask.copy()).unsqueeze(0)

    def _augment(self, img, mask):
        # ── Spatial — applied to image AND mask (kept as before, these are safe) ──
        if random.random() > 0.5:
            img  = img[:, :, ::-1].copy()
            mask = mask[:, ::-1].copy()
        if random.random() > 0.5:
            img  = img[:, ::-1, :].copy()
            mask = mask[::-1, :].copy()
        if random.random() > 0.25:
            k    = random.choice([1, 2, 3])
            img  = np.rot90(img,  k, axes=(1, 2)).copy()
            mask = np.rot90(mask, k, axes=(0, 1)).copy()
        if random.random() > 0.6:  # reduced from 0.5 -> fires less often
            h, w      = img.shape[1], img.shape[2]
            crop_size = random.randint(int(h * 0.85), h)  # less aggressive crop (was 0.7)
            row       = random.randint(0, h - crop_size)
            col       = random.randint(0, w - crop_size)
            img       = img[:, row:row+crop_size, col:col+crop_size]
            mask      = mask[row:row+crop_size, col:col+crop_size]
            img  = np.stack([cv2.resize(img[b], (w, h), interpolation=cv2.INTER_LINEAR) for b in range(img.shape[0])])
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

        # ── Radiometric — REDUCED intensity, image only ──
        if random.random() > 0.6:  # reduced from 0.5
            for b in range(img.shape[0]):
                br = np.random.uniform(0.9, 1.1)   # reduced from (0.8, 1.2) -> ±10%
                ct = np.random.uniform(0.9, 1.1)   # reduced from (0.8, 1.2)
                mv = img[b].mean()
                img[b] = (img[b] - mv) * ct + mv * br
        if random.random() > 0.6:  # reduced from 0.5
            for b in range(img.shape[0]):
                noise_std = np.random.uniform(0.002, 0.015) * img[b].std()  # reduced from (0.005, 0.03)
                img[b]    = img[b] + np.random.normal(0, noise_std, img[b].shape).astype(np.float32)

        # ── Band-specific — REMOVED entirely (band dropout was too extreme) ──
        # (no band dropout in this version)

        # ── Terrain — softened ──
        if random.random() > 0.85:  # reduced from 0.7 -> fires much less often
            h, w   = img.shape[1], img.shape[2]
            sh     = random.randint(h // 10, h // 5)   # smaller region than before
            sw     = random.randint(w // 10, w // 5)
            sr     = random.randint(0, h - sh)
            sc     = random.randint(0, w - sw)
            img[:, sr:sr+sh, sc:sc+sw] *= np.random.uniform(0.6, 0.85)  # softer than (0.3, 0.7)
        if random.random() > 0.9:  # reduced from 0.8 -> fires much less often
            h, w   = img.shape[1], img.shape[2]
            gh     = random.randint(h // 20, h // 8)
            gw     = random.randint(w // 20, w // 8)
            gr     = random.randint(0, h - gh)
            gc     = random.randint(0, w - gw)
            img[:, gr:gr+gh, gc:gc+gw] *= np.random.uniform(1.1, 1.4)  # softer than (1.3, 2.0)

        img = np.clip(img, 0, img.max() if img.max() > 0 else 1)
        return img, mask

print('WaterChipDatasetV3 defined — REDUCED-INTENSITY on-the-fly augmentation')
print('Changes: jitter ±10% (was ±20%), band dropout REMOVED, softer shadow/glint, lower probabilities')

WaterChipDatasetV3 defined — REDUCED-INTENSITY on-the-fly augmentation
Changes: jitter ±10% (was ±20%), band dropout REMOVED, softer shadow/glint, lower probabilities


In [5]:
# ── Cell 2: WaterChipDatasetV3 — On-the-Fly Augmentation (FINAL) ──
# Loads from ORIGINAL chips only (chips_images/images/).
# Applies V3 augmentation suite randomly each time a chip is loaded.
# No pre-generated dataset needed — variety comes from random augmentation every epoch.

import torch, random, cv2
import numpy as np
import rasterio
from pathlib import Path
from torch.utils.data import Dataset

class WaterChipDatasetV3(Dataset):
    def __init__(self, chip_paths, masks_dir, mean, std, augment=False):
        self.chip_paths = chip_paths
        self.masks_dir  = Path(masks_dir)
        self.mean = torch.tensor(mean).view(-1,1,1)
        self.std  = torch.tensor(std).view(-1,1,1)
        self.augment = augment

    def __len__(self): return len(self.chip_paths)

    def __getitem__(self, idx):
        cp = self.chip_paths[idx]
        mp = self.masks_dir / cp.name
        with rasterio.open(cp) as src:
            image = np.stack([
                src.read(5), src.read(3), src.read(2), src.read(7)
            ]).astype(np.float32)
        with rasterio.open(mp) as src:
            mask = (src.read(1) > 0).astype(np.float32)

        if self.augment:
            image, mask = self._augment(image, mask)

        image = (torch.from_numpy(image) - self.mean) / self.std
        return image, torch.from_numpy(mask.copy()).unsqueeze(0)

    def _augment(self, img, mask):
        # ── Spatial — applied to image AND mask ──
        if random.random() > 0.5:
            img  = img[:, :, ::-1].copy()
            mask = mask[:, ::-1].copy()
        if random.random() > 0.5:
            img  = img[:, ::-1, :].copy()
            mask = mask[::-1, :].copy()
        if random.random() > 0.25:
            k    = random.choice([1, 2, 3])
            img  = np.rot90(img,  k, axes=(1, 2)).copy()
            mask = np.rot90(mask, k, axes=(0, 1)).copy()
        if random.random() > 0.5:
            h, w      = img.shape[1], img.shape[2]
            crop_size = random.randint(int(h * 0.7), h)
            row       = random.randint(0, h - crop_size)
            col       = random.randint(0, w - crop_size)
            img       = img[:, row:row+crop_size, col:col+crop_size]
            mask      = mask[row:row+crop_size, col:col+crop_size]
            img  = np.stack([cv2.resize(img[b], (w, h), interpolation=cv2.INTER_LINEAR) for b in range(img.shape[0])])
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

        # ── Radiometric — image only ──
        if random.random() > 0.5:
            for b in range(img.shape[0]):
                br = np.random.uniform(0.8, 1.2)
                ct = np.random.uniform(0.8, 1.2)
                mv = img[b].mean()
                img[b] = (img[b] - mv) * ct + mv * br
        if random.random() > 0.5:
            for b in range(img.shape[0]):
                noise_std = np.random.uniform(0.005, 0.03) * img[b].std()
                img[b]    = img[b] + np.random.normal(0, noise_std, img[b].shape).astype(np.float32)

        # ── Band-specific ──
        if random.random() > 0.7:
            img[random.randint(0, img.shape[0] - 1)] = 0.0

        # ── Terrain ──
        if random.random() > 0.7:
            h, w   = img.shape[1], img.shape[2]
            sh     = random.randint(h // 8, h // 3)
            sw     = random.randint(w // 8, w // 3)
            sr     = random.randint(0, h - sh)
            sc     = random.randint(0, w - sw)
            img[:, sr:sr+sh, sc:sc+sw] *= np.random.uniform(0.3, 0.7)
        if random.random() > 0.8:
            h, w   = img.shape[1], img.shape[2]
            gh     = random.randint(h // 16, h // 6)
            gw     = random.randint(w // 16, w // 6)
            gr     = random.randint(0, h - gh)
            gc     = random.randint(0, w - gw)
            img[:, gr:gr+gh, gc:gc+gw] *= np.random.uniform(1.3, 2.0)

        img = np.clip(img, 0, img.max() if img.max() > 0 else 1)
        return img, mask

print('WaterChipDatasetV3 defined — ON-THE-FLY augmentation')
print('Loads from original chips only (no pre-generated dataset)')
print('Augmentations applied randomly each time a chip is loaded')

WaterChipDatasetV3 defined — ON-THE-FLY augmentation
Loads from original chips only (no pre-generated dataset)
Augmentations applied randomly each time a chip is loaded


In [ ]:
# ── Cell 3: Stability Test 0 — BS=64 + Gradient Clipping + Tversky ──
# NOTE: Do not re-run — results saved to results/unet_training_v3/sweep_bs64_test/
#
# Based on evidence from 3 prior runs (original, reduced-aug, diagnostic):
# ALL configurations collapsed into a near-zero-probability degenerate state,
# always at or shortly after LR reached its peak (0.001). This points to
# LR=0.001 being too aggressive for this Dice-loss + WD=0.1 combination,
# regardless of augmentation intensity.
#
# This run tests two changes simultaneously (given time constraints):
# 1. Lower peak LR: 1e-3 -> 4e-4 (60% reduction)
# 2. Tversky loss instead of Dice (different gradient behaviour on imbalanced data,
#    alpha=0.3/beta=0.7 favours recall, may be more stable than Dice here)
#
# Same warm-up (3 epochs), same smoothing (smooth=1.0), same on-the-fly augmentation
# (original intensity, since reduced-aug did NOT prevent collapse either).

import segmentation_models_pytorch as smp
import torch, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import KFold
from collections import defaultdict
from tqdm import tqdm
import rasterio

BASE_DIR   = Path('/mnt/batch/tasks/shared/LS_root/mounts/clusters/v-lmarotti1/code/Users/v-lmarotti/OmniWaterMask')
CHIPS_DIR  = BASE_DIR / 'images/chips_images/images'
MASKS_DIR  = BASE_DIR / 'images/chips_masks/masks'
OUTPUT_DIR = BASE_DIR / 'results/unet_training_v3/sweep_bs64_test'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IN_CHANNELS   = 4
MAX_EPOCHS    = 100
PATIENCE      = 15
SEED          = 42
N_FOLDS       = 5
LAND_PCT      = 0.20
WARMUP_EPOCHS = 3
DIAG_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]

random.seed(SEED)
np.random.seed(SEED)

with open(BASE_DIR / 'results/unet_training/normalisation_stats.json') as f:
    stats = json.load(f)
mean = np.array(stats['mean'], dtype=np.float32)
std  = np.array(stats['std'],  dtype=np.float32)

TEST_SCENES = {
    '12AUG30193433_sub2', '15JUL02193_sub_2', '16Sep2219_1b_Chiwawa',
    '16Sep2219_1b_Teanaway', 'UGR_2017_08_30_103001007046C600_AOI_1',
    'UGR_2021_07_21_10300100C2972100_AOI_1', 'Wenatchee_2014_08_27_10300100354F7A00_AOI',
    'Wenatchee_2016_08_16_10400100214F5200_AOI_1', 'Wenatchee_2019_07_26_104001004E554F00_AOI_1',
    'Wenatchee_2022_11_09_104001007E867900_AOI_1', 'Wenatchee_2025_02_07_103001010D0A1600_AOI_1',
    'cath_creek_2015_10_09_10300100496A4600_AOI_1', 'cath_creek_2018_12_05_1040010046BF9000_AOI_1',
}

def get_scene_id(filename):
    name = Path(filename).stem
    if '_clipped_' in name: return name.split('_clipped_')[0]
    if '_clip_' in name:    return name.split('_clip_')[0]
    return name

all_chips  = sorted(CHIPS_DIR.glob('*.tif'))
mask_stems = {m.stem for m in MASKS_DIR.glob('*.tif')}

scene_to_water = defaultdict(list)
scene_to_land  = defaultdict(list)

for chip in all_chips:
    if chip.stem not in mask_stems: continue
    sid = get_scene_id(chip.name)
    if sid in TEST_SCENES: continue
    with rasterio.open(MASKS_DIR / chip.name) as src:
        has_water = (src.read(1) > 0).any()
    if has_water:
        scene_to_water[sid].append(chip)
    else:
        scene_to_land[sid].append(chip)

scene_list = sorted(set(scene_to_water.keys()) | set(scene_to_land.keys()))
random.shuffle(scene_list)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold1_train_idx, fold1_val_idx = next(iter(kf.split(scene_list)))
train_scenes = set(scene_list[i] for i in fold1_train_idx)
val_scenes   = set(scene_list[i] for i in fold1_val_idx)

print(f'Train scenes: {len(train_scenes)}, Val scenes: {len(val_scenes)}')

train_water = [c for s in train_scenes for c in scene_to_water.get(s, [])]
train_land_all = [c for s in train_scenes for c in scene_to_land.get(s, [])]
n_land = round(len(train_water) * LAND_PCT / (1 - LAND_PCT))
n_land = min(n_land, len(train_land_all))
train_land = random.sample(train_land_all, n_land)
train_chips = train_water + train_land

val_chips = [c for s in val_scenes for c in (scene_to_water.get(s, []) + scene_to_land.get(s, []))]

print(f'Train chips: {len(train_chips):,} (water={len(train_water):,}, land={len(train_land):,})')
print(f'Val chips:   {len(val_chips):,}')

def get_loss_fn(loss_name):
    dice = smp.losses.DiceLoss(mode='binary', from_logits=True, smooth=1.0)
    if loss_name == 'dice':
        return lambda p, t: dice(p, t)
    elif loss_name == 'tversky':
        tversky = smp.losses.TverskyLoss(mode='binary', from_logits=True, alpha=0.3, beta=0.7, smooth=1.0)
        return lambda p, t: tversky(p, t)
    elif loss_name == 'focal_dice':
        focal = smp.losses.FocalLoss(mode='binary')
        return lambda p, t: 0.5 * dice(p, t) + 0.5 * focal(p, t)

def compute_metrics(preds, targets, threshold=0.3):
    pb = (torch.sigmoid(preds) > threshold).cpu().numpy()
    tb = targets.cpu().numpy()
    water_mask = tb.sum(axis=(1,2,3)) > 0
    if water_mask.sum() == 0:
        return {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
    pb = pb[water_mask].flatten().astype(int)
    tb = tb[water_mask].flatten().astype(int)
    return {
        'f1':        round(f1_score(tb, pb, zero_division=0), 4),
        'precision': round(precision_score(tb, pb, zero_division=0), 4),
        'recall':    round(recall_score(tb, pb, zero_division=0), 4),
    }

def compute_multi_threshold_f1(preds, targets, thresholds):
    probs = torch.sigmoid(preds).cpu().numpy()
    tb    = targets.cpu().numpy()
    water_mask = tb.sum(axis=(1,2,3)) > 0
    results = {}
    if water_mask.sum() == 0:
        for t in thresholds:
            results[t] = 0.0
        prob_stats = {'prob_min': 0, 'prob_max': 0, 'prob_mean': 0, 'prob_std': 0}
        return results, prob_stats
    probs_w = probs[water_mask]
    tb_w    = tb[water_mask].flatten().astype(int)
    for t in thresholds:
        pb = (probs_w > t).flatten().astype(int)
        results[t] = round(f1_score(tb_w, pb, zero_division=0), 4)
    prob_stats = {
        'prob_min':  round(float(probs_w.min()), 4),
        'prob_max':  round(float(probs_w.max()), 4),
        'prob_mean': round(float(probs_w.mean()), 4),
        'prob_std':  round(float(probs_w.std()), 4),
    }
    return results, prob_stats

def make_warmup_cosine_scheduler(optimiser, warmup_epochs, max_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return 0.1 + 0.9 * (epoch / warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, (max_epochs - warmup_epochs))
        return 0.5 * (1 + np.cos(np.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimiser, lr_lambda)

def train_config(cfg):
    print(f'\n{"="*60}')
    print(f'Config: {cfg["name"]}')
    print(f'  LR={cfg["lr"]}, WD={cfg["wd"]}, Loss={cfg["loss"]}, BS={cfg["bs"]} (warm-up={WARMUP_EPOCHS} epochs)')
    print(f'{"="*60}')

    train_ds = WaterChipDatasetV3(train_chips, MASKS_DIR, mean, std, augment=True)
    val_ds   = WaterChipDatasetV3(val_chips,   MASKS_DIR, mean, std, augment=False)

    train_loader = DataLoader(train_ds, batch_size=cfg['bs'], shuffle=True,
                              num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg['bs'], shuffle=False,
                              num_workers=4, pin_memory=True)

    model = smp.Unet(
        encoder_name='resnet34', encoder_weights='imagenet',
        in_channels=IN_CHANNELS, classes=1, activation=None,
    ).to(DEVICE)

    loss_fn   = get_loss_fn(cfg['loss'])
    optimiser = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    scheduler = make_warmup_cosine_scheduler(optimiser, WARMUP_EPOCHS, MAX_EPOCHS)

    best_val_loss = float('inf')
    no_improve    = 0
    history       = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t_losses, t_preds, t_targets = [], [], []
        for imgs, masks in tqdm(train_loader, desc=f'{cfg["name"]} E{epoch} train', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimiser.zero_grad()
            out  = model(imgs)
            loss = loss_fn(out, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
            optimiser.step()
            t_losses.append(loss.item())
            t_preds.append(out.detach().cpu())
            t_targets.append(masks.detach().cpu())
        scheduler.step()
        current_lr = optimiser.param_groups[0]['lr']
        tm = compute_metrics(torch.cat(t_preds), torch.cat(t_targets))
        del t_preds, t_targets

        model.eval()
        v_losses, v_preds, v_targets = [], [], []
        with torch.no_grad():
            for imgs, masks in tqdm(val_loader, desc=f'{cfg["name"]} E{epoch} val', leave=False):
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                out = model(imgs)
                v_losses.append(loss_fn(out, masks).item())
                v_preds.append(out.cpu())
                v_targets.append(masks.cpu())

        v_preds_cat   = torch.cat(v_preds)
        v_targets_cat = torch.cat(v_targets)
        vm = compute_metrics(v_preds_cat, v_targets_cat)
        multi_f1, prob_stats = compute_multi_threshold_f1(v_preds_cat, v_targets_cat, DIAG_THRESHOLDS)
        del v_preds, v_targets, v_preds_cat, v_targets_cat

        t_loss = round(np.mean(t_losses), 4)
        v_loss = round(np.mean(v_losses), 4)

        history.append({
            'epoch': epoch, 'lr': round(current_lr, 6),
            'train_loss': t_loss, 'val_loss': v_loss,
            'train_f1': tm['f1'], 'val_f1': vm['f1'],
            'val_prec': vm['precision'], 'val_rec': vm['recall'],
            **{f'f1_t{t}': multi_f1[t] for t in DIAG_THRESHOLDS},
            **prob_stats,
        })

        print(f"E{epoch:3d} | LR: {current_lr:.6f} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | "
              f"Train F1: {tm['f1']:.4f} | Val F1: {vm['f1']:.4f} | P: {vm['precision']:.4f} | R: {vm['recall']:.4f}")
        print(f"      Probs[min={prob_stats['prob_min']:.4f} max={prob_stats['prob_max']:.4f} "
              f"mean={prob_stats['prob_mean']:.4f} std={prob_stats['prob_std']:.4f}] | "
              f"F1@thresh: " + ' '.join([f't{t}={multi_f1[t]:.3f}' for t in DIAG_THRESHOLDS]))

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            torch.save(model.state_dict(), OUTPUT_DIR / f'{cfg["name"]}_best.pth')
            print(f'  ✓ New best val loss: {best_val_loss:.4f} — model saved')
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(OUTPUT_DIR / f'{cfg["name"]}_history.csv', index=False)

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss', color='#2E75B6')
    axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='Val Loss',   color='#E8A838')
    axes[0].set_title(f'{cfg["name"]} — Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(hist_df['epoch'], hist_df['train_f1'], label='Train F1', color='#2E75B6')
    axes[1].plot(hist_df['epoch'], hist_df['val_f1'],   label='Val F1',   color='#5DCAA5')
    axes[1].set_title(f'{cfg["name"]} — F1 (t=0.3)'); axes[1].legend(); axes[1].grid(alpha=0.3)
    for t in DIAG_THRESHOLDS:
        axes[2].plot(hist_df['epoch'], hist_df[f'f1_t{t}'], label=f't={t}', alpha=0.8)
    axes[2].set_title(f'{cfg["name"]} — F1 across thresholds'); axes[2].legend(); axes[2].grid(alpha=0.3)
    plt.suptitle(cfg['name'], fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{cfg["name"]}_curves.png', dpi=150, bbox_inches='tight')
    plt.close()

    return {
        'config': cfg['name'], 'lr': cfg['lr'], 'wd': cfg['wd'],
        'loss': cfg['loss'], 'bs': cfg['bs'],
        'best_val_loss': best_val_loss, 'best_val_f1': hist_df['val_f1'].max(),
    }

# ═══ STABILITY TEST — Lower LR + Tversky ═══
print('\n' + '█'*60); print('STABILITY TEST — Batch Size 64 + WD=1e-3 + Gradient Clipping'); print('█'*60)
CONFIGS_TEST = [
    {'name': 'Ctest_lr4e4_wd1e3_bs64_gradclip_tversky', 'lr': 4e-4, 'wd': 1e-3, 'loss': 'tversky', 'bs': 64},
]
test_results = [train_config(cfg) for cfg in CONFIGS_TEST]
test_df = pd.DataFrame(test_results)
test_df.to_csv(OUTPUT_DIR / 'stability_test_summary.csv', index=False)
print(test_df[['config','loss','best_val_loss','best_val_f1']].to_string(index=False))

Train scenes: 40, Val scenes: 11
Train chips: 2,606 (water=2,085, land=521)
Val chips:   2,514

████████████████████████████████████████████████████████████
STABILITY TEST — Batch Size 64 + WD=1e-3 + Gradient Clipping
████████████████████████████████████████████████████████████

Config: Ctest_lr4e4_wd1e3_bs64_gradclip_tversky
  LR=0.0004, WD=0.001, Loss=tversky, BS=64 (warm-up=3 epochs)


Ctest_lr4e4_wd1e3_bs64_gradclip_tversky E1 train:  98%|█████████▊| 40/41 [03:33<00:03,  3.07s/it]/anaconda/envs/owm/lib/python3.12/site-packages/torch/autograd/graph.py:744: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
                                                                                                 

E  1 | LR: 0.000160 | Train Loss: 0.7400 | Val Loss: 0.3140 | Train F1: 0.1729 | Val F1: 0.1575 | P: 0.0855 | R: 0.9960
      Probs[min=0.0001 max=1.0000 mean=0.4639 std=0.2028] | F1@thresh: t0.1=0.152 t0.2=0.152 t0.3=0.158 t0.4=0.276 t0.5=0.419
  ✓ New best val loss: 0.3140 — model saved


E  2 | LR: 0.000280 | Train Loss: 0.6233 | Val Loss: 0.3135 | Train F1: 0.3106 | Val F1: 0.4033 | P: 0.2847 | R: 0.6913
      Probs[min=0.0057 max=1.0000 mean=0.2434 std=0.1740] | F1@thresh: t0.1=0.152 t0.2=0.359 t0.3=0.403 t0.4=0.501 t0.5=0.522
  ✓ New best val loss: 0.3135 — model saved


E  3 | LR: 0.000400 | Train Loss: 0.5338 | Val Loss: 0.3346 | Train F1: 0.4225 | Val F1: 0.1818 | P: 0.1005 | R: 0.9512
      Probs[min=0.0064 max=1.0000 mean=0.7074 std=0.3396] | F1@thresh: t0.1=0.155 t0.2=0.159 t0.3=0.182 t0.4=0.185 t0.5=0.188


...

E 35 | LR: 0.000302 | Train Loss: 0.1256 | Val Loss: 0.1094 | Train F1: 0.8736 | Val F1: 0.7937 | P: 0.7324 | R: 0.8663
      Probs[min=0.0000 max=1.0000 mean=0.0972 std=0.2948] | F1@thresh: t0.1=0.792 t0.2=0.793 t0.3=0.794 t0.4=0.794 t0.5=0.795


E 36 | LR: 0.000296 | Train Loss: 0.1101 | Val Loss: 0.0717 | Train F1: 0.8859 | Val F1: 0.8627 | P: 0.8901 | R: 0.8370
      Probs[min=0.0000 max=1.0000 mean=0.0772 std=0.2655] | F1@thresh: t0.1=0.862 t0.2=0.863 t0.3=0.863 t0.4=0.863 t0.5=0.863


E 37 | LR: 0.000291 | Train Loss: 0.1168 | Val Loss: 0.0848 | Train F1: 0.8814 | Val F1: 0.8448 | P: 0.8714 | R: 0.8197
      Probs[min=0.0000 max=1.0000 mean=0.0773 std=0.2657] | F1@thresh: t0.1=0.845 t0.2=0.845 t0.3=0.845 t0.4=0.845 t0.5=0.845


E 38 | LR: 0.000285 | Train Loss: 0.1162 | Val Loss: 0.2719 | Train F1: 0.8798 | Val F1: 0.4865 | P: 0.9516 | R: 0.3268
      Probs[min=0.0000 max=1.0000 mean=0.0280 std=0.1620] | F1@thresh: t0.1=0.496 t0.2=0.490 t0.3=0.486 t0.4=0.483 t0.5=0.480


Ctest_lr4e4_wd1e3_bs64_gradclip_tversky E39 val:  32%|███▎      | 13/40 [01:33<04:23,  9.76s/it]  